In [ ]:
from collections import Counter, OrderedDict, defaultdict
from typing import Dict, Any, Sequence, Tuple, Optional, List

import numpy as np

import os
import sys

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

import Common.config as config
import Common.datatypes as datatypes
import Common.utils as utils

import importlib

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(utils)

In [ ]:
class LruPolicy(datatypes.CachePolicy):
    def __init__(self, max_size: int):
        self.cur_size = 0
        self.max_size = max_size
        self.cache = {}             # (vid, layer, tile) -> (value, size)
        self.access_order = []      # keys in order of access (oldest first)

    def get(self, key):
        if key not in self.cache:
            return None

        self.access_order.remove(key)
        self.access_order.append(key)
        
        return self.cache[key][0]

    def put(self, key, value, size: int):
        evicted = []

        if key in self.cache:
            old_size = self.cache[key][1]
            self.cur_size -= old_size
            self.access_order.remove(key)
            del self.cache[key]

        while self.cur_size + size > self.max_size and self.access_order:
            lru_key = self.access_order.pop(0)  # Remove oldest (least recently used)
            lru_size = self.cache[lru_key][1]
            self.cur_size -= lru_size
            del self.cache[lru_key]
            evicted.append(lru_key)

        # Add new item if there's space
        if self.cur_size + size <= self.max_size:
            self.cache[key] = (value, size)
            self.access_order.append(key)
            self.cur_size += size

        return evicted

    def contains(self, key) -> bool:
        return key in self.cache
    
    def remove(self, key):
        if key not in self.cache:
            return False
        
        size = self.cache[key][1]
        self.cur_size -= size
        self.access_order.remove(key)
        del self.cache[key]

        return True
    
    def clear(self):
        self.cache.clear()
        self.access_order.clear()
        self.cur_size = 0
    
    def get_stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }
    
    def keys(self):
        return self.cache.keys()
    
    def stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }

In [ ]:
class LfuPolicy(datatypes.CachePolicy):
    def __init__(self, max_size: int):
        self.cur_size = 0
        self.max_size = max_size
        self.cache: Dict[Tuple[int, int, int, int], Tuple[Any, int]] = {}
        self.freq: Dict[Tuple[int, int, int, int], int] = {}
        self.freq_to_keys: Dict[int, OrderedDict] = defaultdict(OrderedDict)
        self.min_freq = 0

    def _increment_freq(self, key):
        f = self.freq.get(key, 0)
        if f in self.freq_to_keys and key in self.freq_to_keys[f]:
            # Remove from current frequency bucket
            self.freq_to_keys[f].pop(key, None)
            if not self.freq_to_keys[f]:
                del self.freq_to_keys[f]
                if self.min_freq == f:
                    self.min_freq = f + 1
        # Add to next frequency bucket
        nf = f + 1
        self.freq[key] = nf
        self.freq_to_keys[nf][key] = None

    def get(self, key):
        if key not in self.cache:
            return None
        self._increment_freq(key)
        return self.cache[key][0]

    def _evict_one(self):
        if not self.freq_to_keys:
            return None
        # Ensure min_freq points to an existing bucket
        if self.min_freq not in self.freq_to_keys:
            if self.freq_to_keys:
                self.min_freq = min(self.freq_to_keys.keys())
            else:
                return None
        # Evict least-recently used within the minimum frequency bucket
        victim_key, _ = self.freq_to_keys[self.min_freq].popitem(last=False)
        if not self.freq_to_keys[self.min_freq]:
            del self.freq_to_keys[self.min_freq]
        victim_size = self.cache[victim_key][1]
        self.cur_size -= victim_size
        del self.cache[victim_key]
        self.freq.pop(victim_key, None)
        return victim_key

    def put(self, key, value, size: int):
        evicted = []

        if key in self.cache:
            # Adjust size; treat as an access as well
            old_size = self.cache[key][1]
            self.cur_size -= old_size
            # Update payload before incrementing frequency
            self.cache[key] = (value, size)
            # Bring key to higher freq (counts as access)
            self._increment_freq(key)
        else:
            # New key starts with freq=1; set up before capacity adjustments
            self.cache[key] = (value, size)
            self.freq[key] = 0  # will become 1 after increment
            self.min_freq = 1 if self.min_freq in (0, 1) else min(self.min_freq, 1)
            self._increment_freq(key)

        # Evict until it fits
        while self.cur_size + size > self.max_size:
            victim = self._evict_one()
            if victim is None:
                break
            evicted.append(victim)

        # Add size if it fits; otherwise revert new item
        if self.cur_size + size <= self.max_size:
            self.cur_size += size
        else:
            # Could not fit: remove the just-updated/inserted key
            # Clean up from structures
            f = self.freq.pop(key, None)
            if f is not None and f in self.freq_to_keys and key in self.freq_to_keys[f]:
                self.freq_to_keys[f].pop(key, None)
                if not self.freq_to_keys[f]:
                    del self.freq_to_keys[f]
            # Remove from cache
            if key in self.cache:
                del self.cache[key]
            # Recompute min_freq
            if self.freq_to_keys:
                self.min_freq = min(self.freq_to_keys.keys())
            else:
                self.min_freq = 0

        return evicted

    def contains(self, key) -> bool:
        return key in self.cache

    def remove(self, key):
        if key not in self.cache:
            return False
        size = self.cache[key][1]
        self.cur_size -= size
        del self.cache[key]
        f = self.freq.pop(key, None)
        if f is not None and f in self.freq_to_keys:
            self.freq_to_keys[f].pop(key, None)
            if not self.freq_to_keys[f]:
                del self.freq_to_keys[f]
        if self.freq_to_keys:
            self.min_freq = min(self.freq_to_keys.keys())
        else:
            self.min_freq = 0
        return True

    def clear(self):
        self.cache.clear()
        self.freq.clear()
        self.freq_to_keys.clear()
        self.cur_size = 0
        self.min_freq = 0

    def keys(self):
        return self.cache.keys()

    def get_stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }

    def stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }

In [ ]:
class DrlPolicy(datatypes.CachePolicy):
    def __init__(self, capacity: int, cfg: Any = None):
        self.cfg = cfg
        self.capacity = self.cfg.cache_size
        self.cur_size = 0

        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def get(self, key: datatypes.CacheKey) -> Any:
        return self.cache.get(key, None)
    
    def put(self, key: int, value: Any, size: int) -> list:
        """
        key   -> slot index
        value -> (video_id, tiles)
        size  -> fixed as 1 slot
        """
        evicted = []
        slot = key
        new_video, new_tiles = value
        
        old_video = self.video_idx[slot]
        old_tiles = self.tile_idx[slot]

        if old_video != -1:
            evicted.append((old_video, old_tiles))

        self.video_idx[slot] = new_video
        self.tile_idx[slot] = new_tiles

        self.cur_size = sum(1 for v in self.video_idx if v != -1)

        return evicted

    def contains(self, key: datatypes.CacheKey) -> bool:
        return key in self.video_idx

    def remove(self, key: datatypes.CacheKey) -> bool:
        if key in self.video_idx:
            idx = self.video_idx.index(key)
            self.video_idx[idx] = -1
            self.tile_idx[idx] = [-1] * self.cfg.viewport
            self.cur_size = sum(1 for v in self.video_idx if v != -1)
            return True
        return False

    def clear(self) -> None:
        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]
        self.cur_size = 0

    def keys(self):
        return self.video_idx
    
    def get_capacity(self) -> int:
        return self.cur_size
    
    def update_size(self):
        self.cur_size = sum(1 for v in self.video_idx if v != -1)

    def stats(self) -> Dict[str, Any]:
        return {
            'current_size': self.cur_size,
            'capacity': self.capacity,
            'num_items': len([v for v in self.video_idx if v != -1])
        }

In [ ]:
class CacheEngineEnv:
    def __init__(
        self,
        n_users: int = 1000,
        n_tiles: int = 16,
        n_layers: int = 2,
        n_gops: int = 60,
        n_videos: int = 100,
        cache_capacity: float = 100e6,  # capacity in bytes
        policy: datatypes.CachePolicy = None
    ):
        self.cfg = config.Config 

        self.id = "cacheenv"
        self.max_capacity = cache_capacity
        self.tile_size_bytes = {
            0: 2.0e+6 / n_tiles,  # base layer tile size in bytes
            1: 1.5e+7 / n_tiles   # enhancement layer tile size in bytes
        }
        self.n_users = n_users
        self.n_layers = n_layers
        self.n_videos = n_videos
        self.n_gops = n_gops
        self.n_tiles = n_tiles

        # Initialize LRU policy
        self.policy = policy

        self.cache_bitmap = np.zeros(
            (self.n_videos, self.n_layers, self.n_gops, self.n_tiles), dtype=np.int8
        )

        self.content_popularity = np.zeros((n_videos,), dtype=np.float32)
        self.user_visited = np.zeros((n_users, n_videos), dtype=bool)

    def _cache_base_layer(self, vid_idx, layer_idx):
        key = (vid_idx, layer_idx, None, None)
        bs_layer_size = self.tile_size_bytes[layer_idx] * self.n_tiles * self.n_gops

        evicted = self.policy.put(key, None, int(bs_layer_size))
        
        # Update cache bitmap
        self.cache_bitmap[vid_idx, layer_idx, :, :] = 1
        for e_vid, _, _, _ in evicted:
            self.cache_bitmap[e_vid, :, :, :] = 0

    def _cache_tile(self, vid_idx, layer_idx, tile_idx, gop_idx):
        key = (vid_idx, layer_idx, tile_idx, gop_idx)
        tile_size = self.tile_size_bytes[layer_idx] 

        evicted = self.policy.put(key, None, int(tile_size))

        # Update cache bitmap
        self.cache_bitmap[vid_idx, layer_idx, tile_idx, gop_idx] = 1
        for e_vid, e_layer, e_tile, e_gop in evicted:
            self.cache_bitmap[e_vid, e_layer, e_tile, e_gop] = 0

    def _cache_tile_gop_lru(self, vid_idx: int, layer_idx: int, tile_idx: int, gop_idx: int):
        key = (vid_idx, layer_idx, tile_idx, gop_idx)
        tile_size = self.tile_size_bytes[layer_idx] * self.n_gops

        evicted = self.policy.put(key, None, int(tile_size))

        # Update cache bitmap
        self.cache_bitmap[vid_idx, layer_idx, tile_idx, gop_idx] = 1
        for e_vid, e_layer, e_tile, e_gop in evicted:
            self.cache_bitmap[e_vid, e_layer, e_tile, e_gop] = 0

    def _cache_tile_all_gops_lru(self, vid_idx: int, layer_idx: int, tile_idx: int):
        key = (vid_idx, layer_idx, tile_idx, None)
        tile_size = self.tile_size_bytes[layer_idx] * self.n_gops

        evicted = self.policy.put(key, None, int(tile_size))

        # Update cache bitmap
        self.cache_bitmap[vid_idx, layer_idx, tile_idx, :] = 1
        for e_vid, e_layer, e_tile, _ in evicted:
            self.cache_bitmap[e_vid, e_layer, e_tile, :] = 0

    def _clear_cache(self):
        self.policy.clear()

    def _is_video_cached(self, vid_id: int, layer_id: int = 0) -> bool:
        return np.any(self.cache_bitmap[vid_id, layer_id, :, :] == 1)

    def _is_tile_cached(self, vid_id: int, layer_id: int, tile_id: int, gop_id: int) -> bool:
        return self.cache_bitmap[vid_id, layer_id, tile_id, gop_id] == 1

    def check_tile_in_cache(self, vid_id: int, layer_id: int, tile_id: int, gop_id: int) -> bool:
        return self._is_tile_cached(vid_id, layer_id, tile_id, gop_id)

    def get_video_cache_idx(self, vid):
        for idx, v in enumerate(self.policy.cache):
            if v == (vid, -1):
                return idx
        return -1

    def get_video_cache_idx_pantelis(self, vid):
        for idx, v in enumerate(self.policy.video_idx):
            if v == vid:
                return idx
        return -1

    def update_content_popularity(self, requests):
        for req in requests:
            user = req['u']
            vid = req['video']

            if self.user_visited[user, vid]:
                continue

            self.content_popularity[vid] += 1
            self.user_visited[user, vid] = True

    def get_content_popularity(self):
        return self.content_popularity

    def get_video_popularity_norm(self, vid_id: int) -> float:
        total_requests = np.sum(self.content_popularity)
        
        # Avoid division by zero at the start of the episode
        if total_requests == 0:
            return 0.0
            
        # Return probability: P(v) = count(v) / total_count
        return self.content_popularity[vid_id] / total_requests

    def get_tile(self, vid, layer, tile_idx, gop_idx):
        key = (vid, layer, tile_idx, gop_idx)
        if self.policy.contains(key):
            self.policy.get(key)  # Update access order
            return True
        return False
    
    def get_current_capacity(self):
        return self.policy.get_capacity()
    
    def get_cache_bitmap(self):
        return self.cache_bitmap

    def lru_prefetching(self, action):

        if action is None or action['gop'] >= self.n_gops:
            return self.get_cache_bitmap()

        gop = action['gop']
        vid = action['video']
        tiles = action['tiles']
        base_req_init = action['base_req_init']

        if base_req_init:
            self._cache_base_layer(vid, 0)

        if not self._is_video_cached(vid, 0):
            return self.get_cache_bitmap()

        for tile_idx, tile in enumerate(tiles):
            self._cache_tile(vid, 1, tile_idx, gop)

        return self.get_cache_bitmap()

    def lru_live_prefetching(self, action):
        video = action['video']
        viewport = action['tiles']
        base_layer_init = action['base_req_init']

        # 1) Base layer caching (only when base_req_init is True)
        if base_layer_init:
            self.policy.put(0, (video, list(viewport)), 1)

        # 2) Enhancement tiles for gop > 0 only if base layer is cached
        else:
            vid_idx = self.get_video_cache_idx(video)
            for idx, tile in enumerate(viewport):
                self.policy.tile_idx[vid_idx][idx] = tile

        return self.get_cache_bitmap()

    def drl_prefetching(self, action):
        if action is None or action['action_idx'] == 0:
            return self.get_cache_bitmap()

        video_id = action['video']
        viewport = action['tiles']

        # 1) Base layer caching (only when base_req_init is True)
        if action['base_req_init']:
            action_idx = action['action_idx']
            video_slot = action_idx - 1

            evicted = self.policy.put(video_slot, (video_id, list(viewport)), 1)
            for e_vid, e_tiles in evicted:
                self.cache_bitmap[e_vid, :, :, :] = 0

            self.cache_bitmap[video_id, 0, :, :] = 1

        # 2) Enhancement tiles for gop > 0 only if base layer is cached
        if self._is_video_cached(video_id):
            self.cache_bitmap[video_id, 1, :, :] = 0
            self.cache_bitmap[video_id, 1, viewport, :] = 1

        return self.get_cache_bitmap()

    def drl_prefetching_pantelis(self, action):

        video = action['video']
        viewport = action['tiles']
        action_idx = action['action_idx']
        base_layer_init = action['base_req_init']

        if action_idx == 0:
            return self.get_cache_bitmap()

        # 1) Base layer caching (only when base_req_init is True)
        if base_layer_init:
            video_slot = action_idx - 1

            _ = self.policy.put(video_slot, (video, list(viewport)), 1)

        # 2) Enhancement tiles for gop > 0 only if base layer is cached
        else:
            vid_idx = self.get_video_cache_idx_pantelis(video)
            pos_k = action_idx - (self.cfg.cache_size + vid_idx * 4 + 1)

            _ = self.policy.tile_idx[vid_idx][pos_k]
            self.policy.tile_idx[vid_idx][pos_k] = viewport[0]

        return self.get_cache_bitmap()

    def drl_prefetching_focus(self, action):

        video = action['video']
        viewport = action['tiles']
        action_idx = action['action_idx']
        base_layer_init = action['base_req_init']
        slot = action_idx - 1

        if action_idx == 0:
            return self.get_cache_bitmap()
        
        # 1) Base layer caching (only when base_req_init is True)
        if base_layer_init:
            self.policy.put(slot, (video, -1), 1)

        # 2) Enhancement tiles for gop > 0 only if base layer is cached
        else:
            self.policy.put(slot, (video, viewport[0]), 1)

        return self.get_cache_bitmap()

    def drl_prefetching_dudu(self, action):

        video = action['video']
        viewport = action['tiles']
        action_idx = action['action_idx']
        base_layer_request = action['base_layer_req']

        if action_idx == 0:
            return self.get_cache_bitmap()

        action_idx = action_idx - 1
        
        if base_layer_request:
            _ = self.policy.put(action_idx, (video, list(viewport)), 1)        
        else:
            vid_idx = self.get_video_cache_idx(video)
            self.policy.tile_idx[vid_idx][action_idx] = viewport[0]
    
        return self.get_cache_bitmap()


    def is_full(self):
        return self.policy.get_capacity() >= self.policy.capacity

    def reset(self, **kwargs):
        self._clear_cache()

        self.content_popularity = np.zeros((self.n_videos,), dtype=np.float32)
        self.user_visited = np.zeros((self.n_users, self.n_videos), dtype=bool)

        self.cache_bitmap = np.zeros(
            (self.n_videos, self.n_layers, self.n_tiles, self.n_gops), dtype=np.int8
        )

        ### Randomly pre-fill cache ###
        # rng = np.random.default_rng()
        # keys = [(v, l, t) for v in range(self.n_videos)
        #                  for l in range(self.n_layers)
        #                  for t in range(self.n_tiles)]
        # rng.shuffle(keys)

        # for vid, layer, tile_id in keys:
        #     tile_size = int(self.tile_size_bytes[layer])
        #     if self.policy.cur_size + tile_size > self.max_capacity:
        #         break

        #     # Random GOP index for each tile
        #     gop_idx = rng.integers(0, self.n_gops)
        #     if not self.policy.contains((vid, layer, tile_id, gop_idx)):
        #         self._cache_tile(vid, layer, tile_id, gop_idx)
        ### End Randomly pre-fill cache ###

        return None, {"cache": self.get_cache_bitmap()}


In [7]:
if __name__ == "__main__":
    # Test configuration
    n_tiles = 4
    n_layers = 2
    n_videos = 10
    cache_capacity = 10e6  # 10 MB
    
    print("=" * 5, "Cache Storage Test with LRU Policy (Computed Matrix)", "=" * 5)
    
    # Initialize cache environment with LRU
    cache = CacheEngineEnv(
        n_tiles=n_tiles * n_tiles,
        n_layers=n_layers,
        n_videos=n_videos,
        cache_capacity=cache_capacity
    )
    
    cache.reset()

===== Cache Storage Test with LRU Policy (Computed Matrix) =====


In [8]:
# Test LRU Policy with tuple keys (video, layer, tile)
if __name__ == "__main__":
    print("=" * 5, "LRU Cache Policy Test", "=" * 5)
    
    # Create LRU cache with 10 MB capacity
    lru = LruPolicy(max_size=10 * 1024 * 1024)  # 10 MB
    
    print(f"Initial state: {lru.get_stats()}\n")
    
    # Add some items using tuple keys (video, layer, tile)
    print("Adding items:")
    evicted = lru.put(
        (0, 0, 0), 
        "data_v0_l0_t0", 
        2 * 1024 * 1024
    )  # 2 MB
    print(f"  Added (0,0,0) (2 MB), evicted: {evicted}")
    
    evicted = lru.put(
        (1, 0, 0), 
        "data_v1_l0_t0", 
        3 * 1024 * 1024
    )  # 3 MB
    print(f"  Added (1,0,0) (3 MB), evicted: {evicted}")
    
    evicted = lru.put(
        (2, 0, 0), 
        "data_v2_l0_t0", 
        4 * 1024 * 1024
    )  # 4 MB
    print(f"  Added (2,0,0) (4 MB), evicted: {evicted}")
    
    print(f"\nCurrent state: {lru.get_stats()}")
    print(f"Access order (oldest→newest): {lru.access_order}\n")
    
    # Access tile (0,0,0) (moves it to most recent)
    print("Accessing (0,0,0)...")
    data = lru.get((0, 0, 0))
    print(f"  Retrieved: {data}")
    print(f"  New access order: {lru.access_order}\n")
    
    # Add item that requires eviction
    print("Adding large item (3 MB) - should evict LRU item...")
    evicted = lru.put((3, 0, 0), "data_v3_l0_t0", 3 * 1024 * 1024)
    print(f"  Evicted: {evicted}")
    print(f"  Current access order: {lru.access_order}")
    print(f"  State: {lru.get_stats()}\n")
    
    # Test contains
    print("Testing contains:")
    for key in [(0, 0, 0), (1, 0, 0), (2, 0, 0), (3, 0, 0)]:
        print(f"  {key}: {lru.contains(key)}")
    
    print("\n" + "=" * 60)

===== LRU Cache Policy Test =====
Initial state: {'size': 0, 'max_size': 10485760, 'num_items': 0, 'utilization': 0.0}

Adding items:


ValueError: not enough values to unpack (expected 4, got 3)

In [ ]:
if __name__ == "__main__":
    # Test configuration
    n_tiles = 4
    n_layers = 2
    n_videos = 10
    cache_capacity = 10e6  # 10 MB
    
    print("=" * 5, "Cache Storage Test with LRU Policy (Computed Matrix)", "=" * 5)
    
    # Initialize cache environment with LRU
    cache = CacheEngineEnv(
        n_tiles=n_tiles * n_tiles,
        n_layers=n_layers,
        n_videos=n_videos,
        cache_capacity=cache_capacity
    )
    
    print(f"Configuration:")
    print(f"  Grid size: {n_tiles}x{n_tiles} = {n_tiles*n_tiles} tiles")
    print(f"  Layers: {n_layers}")
    print(f"  Videos: {n_videos}")
    print(f"  Capacity: {cache_capacity/1e6:.1f} MB")
    print(f"  Base layer tile size: {cache.tile_size_bytes[0]/1e6:.3f} MB")
    print(f"  Enhancement tile size: {cache.tile_size_bytes[1]/1e6:.3f} MB")
    print(f"  Using LRU: {cache.policy is not None}")
    print()
    
    # Create sample cache actions (prefetch decisions)
    action = [
        {
            'video': 0,
            'gop': 0,
            'tiles': np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
            'base_req_init': True
        },
        {
            'video': 1,
            'gop': 2,
            'tiles': np.array([0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
            'base_req_init': True
        },
        {
            'video': 0,
            'gop': 1,
            'tiles': np.array([1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
            'base_req_init': True
        },
    ]
    
    print(f"Processing {len(action)} cache actions...")
    print(f"Video requests: {[act['video'] for act in action]}\n")
    
    # Process cache prefetching
    cache_matrix = cache.lru_prefetching(action[0])
    
    print("Cache Results:")
    current_size = cache.get_current_capacity()
    print(f"  Final capacity used: {current_size/1e6:.2f} MB / {cache.max_capacity/1e6:.1f} MB")
    print(f"  Utilization: {(current_size/cache.max_capacity)*100:.1f}%")
    
    if cache.policy:
        stats = cache.policy.get_stats()
        print(f"  LRU stats: {stats['num_items']} items, {stats['utilization']:.1%} full")
    print()
    
    # Show cache content per video
    print("Cached tiles by video:")
    for vid_idx in range(n_videos):
        base_cached = np.sum(cache_matrix[vid_idx, 0, :, :])
        enh_cached = np.sum(cache_matrix[vid_idx, 1, :, :])
        if base_cached > 0 or enh_cached > 0:
            print(f"  Video {vid_idx}: Base={base_cached}/{n_tiles*n_tiles}, Enh={enh_cached}/{n_tiles*n_tiles}")
            if base_cached > 0:
                print(f"    Base layer tiles: {np.where(cache_matrix[vid_idx, 0, :] == 1)[0].tolist()}")
            if enh_cached > 0:
                print(f"    Enh layer tiles:  {np.where(cache_matrix[vid_idx, 1, :] == 1)[0].tolist()}")
    print()
    
    print("Testing tile access (updates LRU):")
    print(f"  Accessing tile (0, 0, 5): {cache.get_tile(0, 0, 5, 0)}")
    print(f"  Accessing tile (1, 0, 3): {cache.get_tile(1, 0, 3, 2)}")
    print(f"  Accessing non-cached (5, 0, 0): {cache.get_tile(5, 0, 0, 0)}")

===== Cache Storage Test with LRU Policy (Computed Matrix) =====
Configuration:
  Grid size: 4x4 = 16 tiles
  Layers: 2
  Videos: 10
  Capacity: 10.0 MB
  Base layer tile size: 0.125 MB
  Enhancement tile size: 0.938 MB
  Using LRU: True

Processing 3 cache actions...
Video requests: [0, 1, 0]

Cache Results:
  Final capacity used: 9.38 MB / 10.0 MB
  Utilization: 93.8%
  LRU stats: 10 items, 93.8% full

Cached tiles by video:
  Video 0: Base=960/16, Enh=10/16
    Base layer tiles: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [ ]:
# --- DRL Policy Tests (single cell) ---

class _DummyCfg:
    def __init__(self, cache_size: int, viewport: int):
        self.cache_size = cache_size
        self.viewport = viewport


def _make_policy(cache_size=2, viewport=4):
    cfg = _DummyCfg(cache_size=cache_size, viewport=viewport)
    return DrlPolicy(capacity=cache_size, cfg=cfg)


def run_drl_policy_tests():
    # Test put inserts and evicts within slot
    policy = _make_policy(cache_size=2, viewport=4)

    evicted = policy.put(0, (5, [1, 2]), 1)
    assert evicted == []
    assert policy.video_idx[0] == 5
    assert policy.tile_idx[0] == [1, 2]
    assert policy.cur_size == 1

    evicted = policy.put(0, (6, [3]), 1)
    assert evicted == [(5, [1, 2])]
    assert policy.video_idx[0] == 6
    assert policy.tile_idx[0] == [3]
    assert policy.cur_size == 1

    evicted = policy.put(1, (7, [0, 1]), 1)
    assert evicted == []
    assert policy.video_idx[1] == 7
    assert policy.tile_idx[1] == [0, 1]
    assert policy.cur_size == 2

    # Test contains uses video id
    policy = _make_policy(cache_size=2, viewport=4)
    policy.put(0, (10, [0]), 1)
    policy.put(1, (20, [1]), 1)

    assert policy.contains(10) is True
    assert policy.contains(20) is True
    assert policy.contains(99) is False

    # Test remove clears slot and updates size
    policy = _make_policy(cache_size=2, viewport=4)
    policy.put(0, (10, [0, 1]), 1)
    policy.put(1, (20, [2]), 1)

    removed = policy.remove(10)
    assert removed is True
    assert policy.video_idx[0] == -1
    assert policy.tile_idx[0] == [-1, -1, -1, -1]
    assert policy.cur_size == 1

    removed = policy.remove(99)
    assert removed is False

    # Test clear resets state
    policy = _make_policy(cache_size=2, viewport=2)
    policy.put(0, (1, [0]), 1)
    policy.put(1, (2, [1]), 1)

    policy.clear()
    assert policy.video_idx == [-1, -1]
    assert policy.tile_idx == [[-1, -1], [-1, -1]]
    assert policy.cur_size == 0

    # Test stats reflect cache
    policy = _make_policy(cache_size=3, viewport=2)
    policy.put(0, (1, [0]), 1)
    policy.put(2, (3, [1]), 1)

    stats = policy.stats()
    assert stats["current_size"] == 2
    assert stats["capacity"] == 3
    assert stats["num_items"] == 2

    print("All DRL policy tests passed.")


# Run tests
if __name__ == "__main__":
    run_drl_policy_tests()

All DRL policy tests passed.


In [ ]:
# --- DRL Policy Tests (complex) ---


def run_drl_policy_complex_tests():
    # 1) Eviction list preserves previous tiles per slot
    policy = _make_policy(cache_size=3, viewport=5)
    policy.put(0, (1, [0, 2]), 1)
    policy.put(1, (2, [1, 3]), 1)
    policy.put(2, (3, [4]), 1)

    evicted = policy.put(1, (9, [2, 4]), 1)
    assert evicted == [(2, [1, 3])]
    assert policy.video_idx == [1, 9, 3]
    assert policy.tile_idx[1] == [2, 4]

    # 2) Replacing the same slot twice keeps size stable
    before_size = policy.cur_size
    policy.put(1, (10, [0]), 1)
    policy.put(1, (11, [1]), 1)
    assert policy.cur_size == before_size
    assert policy.video_idx[1] == 11
    assert policy.tile_idx[1] == [1]

    # 3) contains uses video ids, not slot indices
    assert policy.contains(11) is True
    assert policy.contains(1) is True
    assert policy.contains(2) is False

    # 4) remove deletes only the first matching video id
    policy = _make_policy(cache_size=3, viewport=4)
    policy.put(0, (7, [0]), 1)
    policy.put(1, (7, [1, 2]), 1)
    policy.put(2, (8, [3]), 1)

    removed = policy.remove(7)
    assert removed is True
    assert policy.video_idx.count(7) == 1
    assert policy.cur_size == 2

    # 5) stats reflect cache after mixed operations
    policy.put(0, (9, [0, 1]), 1)
    policy.put(2, (10, [2]), 1)
    stats = policy.stats()
    assert stats["capacity"] == 3
    assert stats["current_size"] == 3
    assert stats["num_items"] == 3

    print("All DRL policy complex tests passed.")

if __name__ == "__main__":
    run_drl_policy_complex_tests()

All DRL policy complex tests passed.


In [ ]:
# --- DRL Policy Stress Tests (cache size 50/100) ---

import random


def run_drl_policy_stress_tests():
    def validate_state(policy, viewport: int):
        # Basic structural checks
        assert len(policy.video_idx) == policy.capacity
        assert len(policy.tile_idx) == policy.capacity

        for vid, tiles in zip(policy.video_idx, policy.tile_idx):
            assert isinstance(tiles, list)
            if vid == -1:
                # empty slot should be all -1 placeholders
                assert all(t == -1 for t in tiles)
                assert len(tiles) == viewport
            else:
                # filled slot: tiles should be valid indices
                assert all(isinstance(t, int) for t in tiles)
                assert all(0 <= t < viewport for t in tiles)

        # Size invariants
        assert policy.cur_size == sum(1 for v in policy.video_idx if v != -1)
        assert policy.cur_size <= policy.capacity

    def stress_run(cache_size: int, n_ops: int, seed: int = 42):
        random.seed(seed)
        policy = _make_policy(cache_size=cache_size, viewport=4)

        for _ in range(n_ops):
            op = random.random()
            if op < 0.75:
                slot = random.randrange(cache_size)
                video_id = random.randrange(cache_size * 5)
                tiles = random.sample(range(4), k=random.randint(1, 4))
                policy.put(slot, (video_id, tiles), 1)
            else:
                if random.random() < 0.5:
                    video_id = random.randrange(cache_size * 5)
                    policy.remove(video_id)
                else:
                    slot = random.randrange(cache_size)
                    video_id = policy.video_idx[slot]
                    if video_id != -1:
                        policy.remove(video_id)

            validate_state(policy, viewport=4)

        stats = policy.stats()
        assert stats["capacity"] == cache_size
        assert stats["current_size"] <= cache_size
        assert stats["num_items"] <= cache_size

    stress_run(cache_size=50, n_ops=5000, seed=7)
    stress_run(cache_size=100, n_ops=10000, seed=11)

    print("DRL policy stress tests passed for cache sizes 50 and 100.")


if __name__ == "__main__":
    run_drl_policy_stress_tests()

DRL policy stress tests passed for cache sizes 50 and 100.


In [ ]:
# --- CacheEngine DRL Prefetching Tests ---


def run_cacheengine_drl_prefetch_tests():
    class _DummyCfg:
        def __init__(self, cache_size: int, viewport: int):
            self.cache_size = cache_size
            self.viewport = viewport

    cfg = _DummyCfg(cache_size=2, viewport=4)
    policy = DrlPolicy(capacity=2, cfg=cfg)

    cache_env = CacheEngineEnv(
        n_users=1,
        n_tiles=12,
        n_layers=2,
        n_gops=3,
        n_videos=5,
        cache_capacity=10e6,
        policy=policy
    )
    cache_env.reset()

    # 1) base_req_init True + action_idx 0 -> no-op
    action = {
        "video": 1,
        "gop": 0,
        "tiles": [0, 4, 7, 11],
        "base_req_init": True,
        "action_idx": 0
    }
    before = cache_env.get_cache_bitmap().copy()
    cache_env.drl_prefetching(action)
    after = cache_env.get_cache_bitmap()
    assert np.array_equal(before, after)

    # 2) base_req_init True + action_idx 1 -> cache base, update policy lists
    action = {
        "video": 2,
        "gop": 0,
        "tiles": [1, 5, 8, 10],
        "base_req_init": True,
        "action_idx": 1
    }
    cache_env.drl_prefetching(action)

    assert cache_env._is_video_cached(2, 0)
    assert policy.video_idx[0] == 2
    assert policy.tile_idx[0] == [1, 5, 8, 10]

    # 3) gop > 0 without base -> no enhancement cached
    action = {
        "video": 3,
        "gop": 1,
        "tiles": [2, 6, 9, 11],
        "base_req_init": False,
        "action_idx": 0
    }
    cache_env.drl_prefetching(action)
    assert not cache_env._is_video_cached(3, 0)
    assert cache_env.cache_bitmap[3, 1, :, 1].sum() == 0

    # 4) gop > 0 with base cached -> enhancement tiles cached
    action = {
        "video": 2,
        "gop": 1,
        "tiles": [3, 7, 9, 11],
        "base_req_init": False,
    }
    cache_env.drl_prefetching(action)
    assert cache_env.cache_bitmap[2, 1, 3, 1] == 1
    assert cache_env.cache_bitmap[2, 1, 7, 1] == 1

    # 5) Eviction: reusing slot clears previous base layer
    action = {
        "video": 4,
        "gop": 0,
        "tiles": [0, 4, 8, 11],
        "base_req_init": True,
        "action_idx": 1
    }
    cache_env.drl_prefetching(action)
    assert cache_env._is_video_cached(4, 0)
    assert not cache_env._is_video_cached(2, 0)
    assert policy.video_idx[0] == 4

    print("All CacheEngine DRL prefetching tests passed.")


if __name__ == "__main__":
    run_cacheengine_drl_prefetch_tests()

All CacheEngine DRL prefetching tests passed.


In [ ]:
# --- CacheBitmap Consistency Test (Enhancement vs Base) ---
def run_cachebitmap_consistency_test():
    class _DummyCfg:
        def __init__(self, cache_size: int, viewport: int):
            self.cache_size = cache_size
            self.viewport = viewport

    cfg = _DummyCfg(cache_size=1, viewport=4)
    policy = DrlPolicy(capacity=1, cfg=cfg)

    cache_env = CacheEngineEnv(
        n_users=1,
        n_tiles=12,
        n_layers=2,
        n_gops=3,
        n_videos=5,
        cache_capacity=10e6,
        policy=policy
    )
    cache_env.reset()

    # Insert video 0 base + enhancement
    action = {
        "video": 0,
        "gop": 0,
        "tiles": [0, 4, 7, 11],
        "base_req_init": True,
        "action_idx": 1
    }
    cache_env.drl_prefetching(action)

    action = {
        "video": 0,
        "gop": 1,
        "tiles": [0, 4, 7, 11],
        "base_req_init": False
    }
    cache_env.drl_prefetching(action)

    assert cache_env._is_video_cached(0, 0)
    assert cache_env.cache_bitmap[0, 1, :, 1].sum() > 0

    # Evict by inserting video 1 in the same slot
    action = {
        "video": 1,
        "gop": 0,
        "tiles": [1, 5, 8, 10],
        "base_req_init": True,
        "action_idx": 1
    }
    cache_env.drl_prefetching(action)

    # Consistency: if base is gone, enhancement must be cleared too
    assert not cache_env._is_video_cached(0, 0)
    assert cache_env.cache_bitmap[0, 1, :, :].sum() == 0

    print("CacheBitmap consistency test passed.")

if __name__ == "__main__":
    run_cachebitmap_consistency_test()

CacheBitmap consistency test passed.


In [ ]:
# --- CacheEngine DRL Prefetching Tests (Multi Enhancements) ---
def run_cacheengine_drl_prefetch_multi_enh_tests():
    class _DummyCfg:
        def __init__(self, cache_size: int, viewport: int):
            self.cache_size = cache_size
            self.viewport = viewport

    cfg = _DummyCfg(cache_size=2, viewport=4)
    policy = DrlPolicy(capacity=2, cfg=cfg)

    cache_env = CacheEngineEnv(
        n_users=1,
        n_tiles=12,
        n_layers=2,
        n_gops=4,
        n_videos=6,
        cache_capacity=10e6,
        policy=policy
    )
    cache_env.reset()

    # Base init for video 0 in slot 1
    base_action = {
        "video": 0,
        "gop": 0,
        "tiles": [0, 4, 7, 11],
        "base_req_init": True,
        "action_idx": 1
    }
    cache_env.drl_prefetching(base_action)

    assert cache_env._is_video_cached(0, 0)
    assert policy.video_idx[0] == 0
    assert policy.tile_idx[0] == [0, 4, 7, 11]

    # Multiple enhancement prefetches across GOPs
    enh_actions = [
        {"video": 0, "gop": 1, "tiles": [0, 4, 7, 11], "base_req_init": False},
        {"video": 0, "gop": 2, "tiles": [1, 5, 8, 10], "base_req_init": False},
        {"video": 0, "gop": 3, "tiles": [2, 6, 9, 11], "base_req_init": False},
    ]

    for action in enh_actions:
        cache_env.drl_prefetching(action)

    # Bitmap checks: all enhancement tiles written for each GOP
    assert cache_env.cache_bitmap[0, 1, 0, 1] == 1
    assert cache_env.cache_bitmap[0, 1, 4, 1] == 1
    assert cache_env.cache_bitmap[0, 1, 7, 1] == 1
    assert cache_env.cache_bitmap[0, 1, 11, 1] == 1

    assert cache_env.cache_bitmap[0, 1, 1, 2] == 1
    assert cache_env.cache_bitmap[0, 1, 5, 2] == 1
    assert cache_env.cache_bitmap[0, 1, 8, 2] == 1
    assert cache_env.cache_bitmap[0, 1, 10, 2] == 1

    assert cache_env.cache_bitmap[0, 1, 2, 3] == 1
    assert cache_env.cache_bitmap[0, 1, 6, 3] == 1
    assert cache_env.cache_bitmap[0, 1, 9, 3] == 1
    assert cache_env.cache_bitmap[0, 1, 11, 3] == 1

    # Policy consistency: enhancement prefetches should not change slot mapping
    assert policy.video_idx[0] == 0
    assert policy.tile_idx[0] == [0, 4, 7, 11]

    # Sanity: only video 0 should have enhancement cached
    assert cache_env.cache_bitmap[1:, 1, :, :].sum() == 0

    print("CacheEngine DRL multi-enhancement tests passed.")


if __name__ == "__main__":
    run_cacheengine_drl_prefetch_multi_enh_tests()

CacheEngine DRL multi-enhancement tests passed.


In [ ]:
# --- New Tests: same GOP tiles + video replacement (via drl_prefetching) ---

def run_same_gop_multiple_tiles_test():
    class _DummyCfg:
        def __init__(self, cache_size: int, viewport: int):
            self.cache_size = cache_size
            self.viewport = viewport

    cfg = _DummyCfg(cache_size=2, viewport=4)
    policy = DrlPolicy(capacity=2, cfg=cfg)
    cache_env = CacheEngineEnv(
        n_users=1,
        n_tiles=12,
        n_layers=2,
        n_gops=4,
        n_videos=3,
        cache_capacity=10e6,
        policy=policy
    )
    cache_env.reset()

    # Base init for video 0 (slot 0)
    base_action = {
        "video": 0,
        "gop": 0,
        "tiles": [0, 1, 2, 3],
        "base_req_init": True,
        "action_idx": 1
    }
    cache_env.drl_prefetching(base_action)
    assert cache_env._is_video_cached(0, 0)

    # Cache 4 tiles in the same GOP via drl_prefetching
    gop = 0
    tiles_to_cache = [2, 3, 4, 5]
    enh_action = {
        "video": 0,
        "gop": gop,
        "tiles": tiles_to_cache,
        "base_req_init": False
    }
    cache_env.drl_prefetching(enh_action)

    for t in tiles_to_cache:
        assert cache_env.cache_bitmap[0, 1, t, gop] == 1
    assert cache_env.cache_bitmap[0, 1, :, gop].sum() == 4

    print("Same-GOP multiple tiles (max 4) test passed.")
    
def run_video_replacement_test():
    class _DummyCfg:
        def __init__(self, cache_size: int, viewport: int):
            self.cache_size = cache_size
            self.viewport = viewport

    cfg = _DummyCfg(cache_size=1, viewport=4)
    policy = DrlPolicy(capacity=1, cfg=cfg)
    cache_env = CacheEngineEnv(
        n_users=1,
        n_tiles=12,
        n_layers=2,
        n_gops=3,
        n_videos=5,
        cache_capacity=10e6,
        policy=policy
    )
    cache_env.reset()

    # Insert video 0 base layer in slot 0
    action = {
        "video": 0,
        "gop": 0,
        "tiles": [0, 4, 7, 11],
        "base_req_init": True,
        "action_idx": 1
    }
    cache_env.drl_prefetching(action)
    assert cache_env._is_video_cached(0, 0)

    # Replace with video 1 in same slot (evicts video 0)
    action = {
        "video": 1,
        "gop": 0,
        "tiles": [1, 5, 8, 10],
        "base_req_init": True,
        "action_idx": 1
    }
    cache_env.drl_prefetching(action)

    # Old video 0 should be fully cleared
    assert not cache_env._is_video_cached(0, 0)
    assert cache_env.cache_bitmap[0, :, :, :].sum() == 0

    # New video 1 should be cached
    assert cache_env._is_video_cached(1, 0)
    assert policy.video_idx[0] == 1

    print("Video replacement test passed.")


if __name__ == "__main__":
    run_same_gop_multiple_tiles_test()
    run_video_replacement_test()

Checking tile 2...
Checking tile 3...
Checking tile 4...
Checking tile 5...
Same-GOP multiple tiles (max 4) test passed.
Video replacement test passed.
